In [19]:
import torch
import neck
from neck.structs import ShardNDSlices

In [41]:
# to be rewritten
g0 = torch.Generator(device='cuda')
g0.manual_seed(0)
a = torch.rand((1024, 1024), generator=g0, device='cuda', dtype=torch.bfloat16)
b = a + 7.8e-3 * a * torch.randint(-1, 2, a.shape, generator=g0, device='cuda', dtype=torch.bfloat16)
c = a + 7.8e-3 * a * torch.randint(-1, 2, a.shape, generator=g0, device='cuda', dtype=torch.bfloat16)
d = a + 7.8e-3 * a * torch.randint(-1, 2, a.shape, generator=g0, device='cuda', dtype=torch.bfloat16)
pairs = [(a,b), (a,c), (a,d), (b,c), (b,d), (c,d)]

In [43]:
print("Relative error / machine epsilon before rewriting:")
for pair in pairs:
    print(f"{neck.inspector.tensor_rel_err(*pair) / 7.8e-3: .2f}")
print("Proportion of equal elements before rewriting:")
for pair in pairs:
    t1, t2 = pair
    print(f"{(t1 == t2).sum().item() / t1.numel() * 100:.2f}%")

Relative error / machine epsilon before rewriting:
 0.85
 0.85
 0.85
 1.20
 1.20
 1.20
Proportion of equal elements before rewriting:
33.47%
33.39%
33.48%
33.51%
33.50%
33.54%


In [44]:
name = "a"
shard_slice = ShardNDSlices(a.shape)
for t in [a,b,c,d]:
    neck.random.rewrite_low_bits(name, shard_slice, t, 2)

In [45]:
print("Relative error / machine epsilon after rewriting:")
for pair in pairs:
    print(f"{neck.inspector.tensor_rel_err(*pair) / 7.8e-3: .2f}")
print("Proportion of equal elements after rewriting:")
for pair in pairs:
    t1, t2 = pair
    print(f"{(t1 == t2).sum().item() / t1.numel() * 100:.2f}%")

Relative error / machine epsilon after rewriting:
 1.32
 1.32
 1.32
 1.53
 1.53
 1.53
Proportion of equal elements after rewriting:
75.05%
75.06%
75.01%
66.78%
66.70%
66.79%


In [29]:
# for verification
g1 = torch.Generator(device='cuda')
g1.manual_seed(0)
aa = torch.rand((1024, 1024), generator=g1, device='cuda', dtype=torch.bfloat16)
bb = aa + 7.8e-3 * aa * torch.randint(-1, 2, aa.shape, generator=g1, device='cuda', dtype=torch.bfloat16)
cc = aa + 7.8e-3 * aa * torch.randint(-1, 2, aa.shape, generator=g1, device='cuda', dtype=torch.bfloat16)
dd = aa + 7.8e-3 * aa * torch.randint(-1, 2, aa.shape, generator=g1, device='cuda', dtype=torch.bfloat16)

In [31]:
for (t, tt) in zip([a, b, c, d], [aa, bb, cc, dd]):
    assert torch.all(t == tt), f"Mismatch found: {t} vs {tt}"

In [49]:
mask_ab = (a == b)

print("before rewrite, same exponent:")
print(neck.inspector.tensor_rel_err(aa[mask_ab], bb[mask_ab]) / 7.8e-3)

print("before rewrite, different exponent:")
print(neck.inspector.tensor_rel_err(aa[~mask_ab], bb[~mask_ab]) / 7.8e-3)

print("after rewrite, same exponent:")
print(neck.inspector.tensor_rel_err(a[mask_ab], b[mask_ab]) / 7.8e-3)  # should be 0

print("after rewrite, different exponent:")
print(neck.inspector.tensor_rel_err(a[~mask_ab], b[~mask_ab]) / 7.8e-3)


before rewrite, same exponent:
0.7398276123156652
before rewrite, different exponent:
1.0761667558177892
after rewrite, same exponent:
0.0
after rewrite, different exponent:
2.5158018169884357
